In [19]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL A: LOAD ALL DATA FROM KAGGLE DATASETS
# ═══════════════════════════════════════════════════════════════════════════

import os
import pickle
import pandas as pd
from pathlib import Path

# ── PATHS ──────────────────────────────────────────────────────────────────
FINAL_SUBMISSION_DIR = "/kaggle/input/datasets/aditya103856/final-submission"
RETRIEVAL_CACHE_DIR  = "/kaggle/input/datasets/coding1056/retrieval-cache"
ENTITY_DATA_PATH     = "/kaggle/input/datasets/aditya103856/entity-data/entity_data_cache.pkl"

# ══════════════════════════════════════════════════════════════════════════
# 1. LOAD PREDICTION TSVs
# ══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("📂 1. LOADING PREDICTION FILES")
print("=" * 60)

def load_onehot_tsv(filepath):
    """Load one-hot TSV → single predicted letter column."""
    df = pd.read_csv(filepath, sep='\t', dtype=str)
    df.columns = [c.strip() for c in df.columns]
    for col in ['A', 'B', 'C', 'D']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['predicted'] = df[['A', 'B', 'C', 'D']].idxmax(axis=1)
    return df[['id', 'predicted']]

PRED_FILES = {
    'baseline_no_rag'        : 'predictions_baseline_no_rag_onehot.tsv',
    'rag_basic'              : 'predictions_rag_basic_onehot.tsv',
    'phase1_countryfilter'   : 'predictions_phase1_countryfilter_onehot.tsv',
    'phase2_intent'          : 'predictions_phase2_intent_onehot.tsv',
    'phase3_tiered'          : 'predictions_phase3_tiered_onehot.tsv',
    'phase4_quality'         : 'predictions_phase4_quality_onehot.tsv',
    'phase5_trust_weight'    : 'predictions_phase5_trust_weight_onehot.tsv',
    'phase6_full_system'     : 'predictions_phase6_full_system_onehot.tsv',
}

predictions = {}
for name, fname in PRED_FILES.items():
    fpath = os.path.join(FINAL_SUBMISSION_DIR, fname)
    if os.path.exists(fpath):
        predictions[name] = load_onehot_tsv(fpath)
        print(f"  ✅ {name:<30} → {len(predictions[name]):,} rows")
    else:
        print(f"  ⚠️  MISSING: {fname}")

# Load gold answers
ANS_FILE = os.path.join(FINAL_SUBMISSION_DIR, "final_2_mcq_reference.tsv")
if os.path.exists(ANS_FILE):
    ans_df = pd.read_csv(ANS_FILE, sep='\t', dtype=str)
    ans_df.columns = [c.strip() for c in ans_df.columns]
    ans_df['answer'] = ans_df['answer'].str.strip().str.upper()
    gold_map = dict(zip(ans_df['id'], ans_df['answer']))
    print(f"\n  ✅ Gold answers loaded: {len(gold_map):,}")
else:
    print(f"  ❌ MISSING: final_2_mcq_reference.tsv")
    ans_df, gold_map = None, {}

# ══════════════════════════════════════════════════════════════════════════
# 2. LOAD KB CHUNKS (7 shards)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📂 2. LOADING KB CHUNKS")
print("=" * 60)

kb_chunks = []
for i in range(1, 8):
    fpath = os.path.join(RETRIEVAL_CACHE_DIR, f"kb_chunks_{i}.pkl")
    if os.path.exists(fpath):
        with open(fpath, 'rb') as f:
            shard = pickle.load(f)
        kb_chunks.extend(shard)
        print(f"  ✅ kb_chunks_{i}.pkl → {len(shard):,} chunks")
    else:
        print(f"  ⚠️  MISSING: kb_chunks_{i}.pkl")

print(f"\n  📦 Total KB chunks: {len(kb_chunks):,}")

# Quick stats
if kb_chunks:
    countries_in_kb = set()
    for c in kb_chunks:
        ctry = c.get('country', None)
        if ctry:
            countries_in_kb.add(str(ctry).upper())
    print(f"  🌍 Countries covered: {len(countries_in_kb)} → {sorted(countries_in_kb)}")

# ══════════════════════════════════════════════════════════════════════════
# 3. LOAD & EXTRACT ENTITY DATA
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📂 3. LOADING ENTITY DATA")
print("=" * 60)

if not os.path.exists(ENTITY_DATA_PATH):
    print(f"  ❌ MISSING: {ENTITY_DATA_PATH}")
    entity_data, entity_map, df = [], {}, None
else:
    with open(ENTITY_DATA_PATH, 'rb') as f:
        raw_cache = pickle.load(f)

    # Handle both dict-format and list-format cache
    if isinstance(raw_cache, dict):
        entity_data = raw_cache.get('entity_data', [])
        df          = raw_cache.get('df', None)
        print(f"  ✅ Cache format  : dict")
        print(f"  📅 Timestamp     : {raw_cache.get('timestamp', 'unknown')}")
        print(f"  🔑 Keys          : {list(raw_cache.keys())}")
    elif isinstance(raw_cache, list):
        entity_data = raw_cache
        df          = None
        print(f"  ✅ Cache format  : list (legacy)")
    else:
        raise ValueError(f"Unknown cache format: {type(raw_cache)}")

    print(f"  📊 entity_data   : {len(entity_data):,} items")

    # ── Build O(1) lookup map ────────────────────────────────────────────
    entity_map = {item['id']: item for item in entity_data}

    # ── Print statistics ─────────────────────────────────────────────────
    countries  = set(d.get('country') for d in entity_data if d.get('country'))
    has_intent = sum(1 for d in entity_data if d.get('intent'))
    has_opts   = sum(1 for d in entity_data if d.get('options'))
    total_ents = sum(len(d.get('entities', [])) for d in entity_data)

    print(f"  🌍 Countries     : {len(countries)} → {sorted(countries)}")
    print(f"  🏷️  Has intent    : {has_intent}/{len(entity_data)}")
    print(f"  📝 Has options   : {has_opts}/{len(entity_data)}")
    print(f"  🔍 Total entities: {total_ents:,} ({total_ents/len(entity_data):.1f} avg/question)")

    # ── Sample 3 items ───────────────────────────────────────────────────
    print(f"\n  📋 Sample entity_data (first 3):")
    for item in entity_data[:3]:
        print(f"    id       : {item.get('id')}")
        print(f"    country  : {item.get('country')}")
        print(f"    intent   : {item.get('intent', '—')}")
        print(f"    entities : {item.get('entities', [])[:5]}")
        print(f"    options  : {item.get('options', '—')}")
        print()

    # ── If df missing from cache, note it ───────────────────────────────
    if df is None:
        print("  ℹ️  df not in cache (normal — load questions.tsv separately if needed)")
    else:
        print(f"  ✅ df (questions): {len(df):,} rows | cols: {list(df.columns)}")

# ══════════════════════════════════════════════════════════════════════════
# 4. FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("✅ ALL DATA LOADED — GLOBALS AVAILABLE:")
print("=" * 60)
print(f"  predictions   : dict with {len(predictions)} configs")
print(f"  ans_df        : {len(ans_df) if ans_df is not None else 0} gold answers")
print(f"  gold_map      : {len(gold_map)} id→answer pairs")
print(f"  kb_chunks     : {len(kb_chunks):,} KB chunks")
print(f"  entity_data   : {len(entity_data):,} items")
print(f"  entity_map    : {len(entity_map):,} id→item (O(1) lookup)")
print(f"  df            : {'✅ loaded' if df is not None else '❌ not in cache'}")
print("=" * 60)


📂 1. LOADING PREDICTION FILES
  ✅ baseline_no_rag                → 47,014 rows
  ✅ rag_basic                      → 47,014 rows
  ✅ phase1_countryfilter           → 47,014 rows
  ✅ phase2_intent                  → 47,014 rows
  ✅ phase3_tiered                  → 47,014 rows
  ✅ phase4_quality                 → 47,014 rows
  ✅ phase5_trust_weight            → 47,014 rows
  ✅ phase6_full_system             → 47,014 rows

  ✅ Gold answers loaded: 47,014

📂 2. LOADING KB CHUNKS
  ✅ kb_chunks_1.pkl → 158 chunks
  ✅ kb_chunks_2.pkl → 246 chunks
  ✅ kb_chunks_3.pkl → 176 chunks
  ✅ kb_chunks_4.pkl → 176 chunks
  ✅ kb_chunks_5.pkl → 190 chunks
  ✅ kb_chunks_6.pkl → 154 chunks
  ✅ kb_chunks_7.pkl → 162 chunks

  📦 Total KB chunks: 1,262
  🌍 Countries covered: 44 → ['ALGERIA', 'AS', 'AU', 'AZ', 'AZERBAIJAN', 'BG', 'CHINA', 'CN', 'DZ', 'EC', 'EG', 'ES', 'ET', 'ETHIOPIA', 'FR', 'GB', 'GR', 'GREECE', 'ID', 'IE', 'INDONESIA', 'IR', 'IRAN', 'JB', 'JP', 'KP', 'KR', 'LK', 'MA', 'MEXICO', 'MX', 'NG', 'N

In [20]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL B: BUILD MASTER DATAFRAME
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd

print("=" * 65)
print("📐 BUILDING MASTER DATAFRAME")
print("=" * 65)

# ── STEP 1: Questions from df (already in memory from Cell A) ─────────────
if df is not None and 'question' in df.columns:
    keep = [c for c in ['id','question','option_A','option_B','option_C','option_D','answer'] if c in df.columns]
    master_df = df[keep].copy()
    print(f"  ✅ Questions from entity_data cache: {len(master_df):,} rows")
else:
    # Fallback: reconstruct from entity_data options field
    rows = []
    for item in entity_data:
        opts = item.get('options', ['','','',''])
        opts = (opts + ['','','',''])[:4]
        rows.append({'id': item['id'], 'question': item.get('question',''),
                     'option_A': opts[0], 'option_B': opts[1],
                     'option_C': opts[2], 'option_D': opts[3]})
    master_df = pd.DataFrame(rows)
    print(f"  ✅ Questions rebuilt from entity_data: {len(master_df):,} rows")

# ── STEP 2: Merge answers if not already present ──────────────────────────
if 'answer' not in master_df.columns:
    master_df = master_df.merge(ans_df[['id','answer']], on='id', how='inner')
    print(f"  ✅ Answers merged from ans_df")
else:
    print(f"  ✅ Answers already in df")

master_df['answer'] = master_df['answer'].str.strip().str.upper()

# ── STEP 3: Merge country + intent from entity_data ───────────────────────
meta_df = pd.DataFrame([{
    'id':     item['id'],
    'country': item.get('country', 'UNK'),
    'intent':  item.get('intent', 'others'),
} for item in entity_data])

master_df = master_df.merge(meta_df, on='id', how='left')
master_df['country'] = master_df['country'].fillna('UNK')
master_df['intent']  = master_df['intent'].fillna('others')
print(f"  ✅ Country + intent merged")

# ── STEP 4: Merge all 8 predictions + correctness flag ────────────────────
CONFIG_DISPLAY = {
    'baseline_no_rag'     : 'Baseline',
    'rag_basic'           : 'RAG Basic',
    'phase1_countryfilter': 'Ph1 Country',
    'phase2_intent'       : 'Ph2 Intent',
    'phase3_tiered'       : 'Ph3 Tiered',
    'phase4_quality'      : 'Ph4 Quality',
    'phase5_trust_weight' : 'Ph5 Trust',
    'phase6_full_system'  : 'Ph6 Full ✦',
}

for key, label in CONFIG_DISPLAY.items():
    if key not in predictions:
        print(f"  ⚠️  MISSING config: {key}")
        continue
    p = predictions[key].rename(columns={'predicted': f'pred_{key}'})
    master_df = master_df.merge(p, on='id', how='left')
    master_df[f'pred_{key}']    = master_df[f'pred_{key}'].fillna('C')
    master_df[f'correct_{key}'] = master_df[f'pred_{key}'] == master_df['answer']
    acc = master_df[f'correct_{key}'].mean() * 100
    print(f"  ✅ {label:<15} → {master_df[f'correct_{key}'].sum()}/{len(master_df)} ({acc:.1f}%)")

master_df.to_csv('/kaggle/working/master_df.csv', index=False)
print(f"\n{'='*65}")
print(f"✅ master_df ready: {len(master_df):,} rows × {len(master_df.columns)} cols")
print(f"   Countries : {sorted(master_df['country'].unique())}")
print(f"   Intents   : {sorted(master_df['intent'].unique())}")
print(f"{'='*65}")


📐 BUILDING MASTER DATAFRAME
  ✅ Questions from entity_data cache: 47,014 rows
  ✅ Answers merged from ans_df
  ✅ Country + intent merged
  ✅ Baseline        → 36932/47014 (78.6%)
  ✅ RAG Basic       → 36639/47014 (77.9%)
  ✅ Ph1 Country     → 36639/47014 (77.9%)
  ✅ Ph2 Intent      → 36639/47014 (77.9%)
  ✅ Ph3 Tiered      → 36465/47014 (77.6%)
  ✅ Ph4 Quality     → 36928/47014 (78.5%)
  ✅ Ph5 Trust       → 36928/47014 (78.5%)
  ✅ Ph6 Full ✦      → 36928/47014 (78.5%)

✅ master_df ready: 47,014 rows × 25 cols
   Countries : ['AS', 'AU', 'AZ', 'BG', 'CN', 'DZ', 'EC', 'EG', 'ES', 'ET', 'FR', 'GB', 'GR', 'ID', 'IE', 'IR', 'JB', 'JP', 'KP', 'KR', 'LK', 'MA', 'MX', 'NG', 'PH', 'PV', 'SA', 'SE', 'SG', 'US']
   Intents   : ['others']


In [21]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL C: FULL ABLATION LEADERBOARD
# Accuracy, delta, 95% CI, rank, answer-bias detection
# ═══════════════════════════════════════════════════════════════════════════

import math
import pandas as pd

print("=" * 80)
print("🏆 ABLATION LEADERBOARD — ALL 8 CONFIGURATIONS")
print("=" * 80)

CONFIG_DISPLAY = {
    'baseline_no_rag'     : 'Baseline (LLM only)',
    'rag_basic'           : 'RAG Basic (unfiltered)',
    'phase1_countryfilter': '+ Country Filter',
    'phase2_intent'       : '+ Intent Detection',
    'phase3_tiered'       : '+ Tiered Routing',
    'phase4_quality'      : '+ Quality Signals',
    'phase5_trust_weight' : '+ Trust Reranking',
    'phase6_full_system'  : 'Full System ✦',
}

n            = len(master_df)
baseline_acc = None
rows         = []

for key, label in CONFIG_DISPLAY.items():
    col      = f'correct_{key}'
    pred_col = f'pred_{key}'
    if col not in master_df.columns:
        rows.append({'Config': label, 'Correct': '—', 'Acc': '—',
                     '95% CI': '—', 'Δ Baseline': '—', 'Rank': '—',
                     'A%':'—','B%':'—','C%':'—','D%':'—'})
        continue

    correct = int(master_df[col].sum())
    acc     = correct / n * 100

    # Wilson 95% CI
    p      = correct / n
    z      = 1.96
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    margin = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    ci_lo  = max(0,   (centre - margin) * 100)
    ci_hi  = min(100, (centre + margin) * 100)

    if baseline_acc is None:
        baseline_acc = acc
        delta_str    = '—'
        arrow        = ' '
    else:
        delta     = acc - baseline_acc
        arrow     = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
        delta_str = f"{arrow}{delta:+.1f}%"

    dist  = master_df[pred_col].value_counts(normalize=True) * 100
    rows.append({
        'Config'    : label,
        'Correct'   : correct,
        'Acc'       : round(acc, 1),
        '95% CI'    : f"[{ci_lo:.1f},{ci_hi:.1f}]",
        'Δ Baseline': delta_str,
        'A%'        : f"{dist.get('A',0):.0f}",
        'B%'        : f"{dist.get('B',0):.0f}",
        'C%'        : f"{dist.get('C',0):.0f}",
        'D%'        : f"{dist.get('D',0):.0f}",
    })

lb = pd.DataFrame(rows)
numeric = lb[lb['Acc'] != '—'].copy()
numeric['Acc'] = numeric['Acc'].astype(float)
lb.loc[lb['Acc'] != '—', 'Rank'] = (
    numeric['Acc'].rank(ascending=False, method='min').astype(int).astype(str).values
)

# Gold distribution reference
gold = master_df['answer'].value_counts(normalize=True) * 100

print(f"\n  {'Config':<26} {'Corr':>5} {'Acc%':>6} {'95% CI':>14} {'Δ Base':>9} "
      f"{'Rank':>5}  {'A%':>4} {'B%':>4} {'C%':>4} {'D%':>4}")
print("  " + "─" * 88)
for _, r in lb.iterrows():
    print(f"  {r['Config']:<26} {str(r['Correct']):>5} {str(r['Acc']):>6} "
          f"{str(r['95% CI']):>14} {str(r['Δ Baseline']):>9} "
          f"{str(r.get('Rank','—')):>5}  {r['A%']:>4} {r['B%']:>4} {r['C%']:>4} {r['D%']:>4}")

print(f"\n  {'Gold dist (ref)':<26} {'':>5} {'':>6} {'':>14} {'':>9} {'':>5}  "
      f"{gold.get('A',0):>3.0f}% {gold.get('B',0):>3.0f}% "
      f"{gold.get('C',0):>3.0f}% {gold.get('D',0):>3.0f}%")

lb.to_csv('/kaggle/working/leaderboard.csv', index=False)
print(f"\n  📁 Saved: leaderboard.csv")
print("=" * 80)


🏆 ABLATION LEADERBOARD — ALL 8 CONFIGURATIONS

  Config                      Corr   Acc%         95% CI    Δ Base  Rank    A%   B%   C%   D%
  ────────────────────────────────────────────────────────────────────────────────────────
  Baseline (LLM only)        36932   78.6    [78.2,78.9]         —     1    32   25   22   21
  RAG Basic (unfiltered)     36639   77.9    [77.6,78.3]    ▼-0.6%     5    32   24   23   21
  + Country Filter           36639   77.9    [77.6,78.3]    ▼-0.6%     5    32   24   23   21
  + Intent Detection         36639   77.9    [77.6,78.3]    ▼-0.6%     5    32   24   23   21
  + Tiered Routing           36465   77.6    [77.2,77.9]    ▼-1.0%     8    33   24   22   21
  + Quality Signals          36928   78.5    [78.2,78.9]    ▼-0.0%     2    30   25   24   21
  + Trust Reranking          36928   78.5    [78.2,78.9]    ▼-0.0%     2    30   25   24   21
  Full System ✦              36928   78.5    [78.2,78.9]    ▼-0.0%     2    30   25   24   21

  Gold dist (re

In [22]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL D: COUNTRY × CONFIG ACCURACY MATRIX
# Per-country accuracy for all 8 configs + best config + RAG delta
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from collections import Counter

print("=" * 85)
print("🌍 COUNTRY × CONFIG ACCURACY MATRIX")
print("=" * 85)

CONFIG_SHORT = {
    'baseline_no_rag'     : 'BASE',
    'rag_basic'           : 'RAG0',
    'phase1_countryfilter': 'PH1',
    'phase2_intent'       : 'PH2',
    'phase3_tiered'       : 'PH3',
    'phase4_quality'      : 'PH4',
    'phase5_trust_weight' : 'PH5',
    'phase6_full_system'  : 'PH6✦',
}

available = [k for k in CONFIG_SHORT if f'correct_{k}' in master_df.columns]
shorts    = [CONFIG_SHORT[k] for k in available]
countries = sorted(master_df['country'].unique())

matrix_rows = []
for ctry in countries:
    sub = master_df[master_df['country'] == ctry]
    row = {'Country': ctry, 'N': len(sub)}
    accs = {}
    for k in available:
        a = sub[f'correct_{k}'].mean() * 100
        row[CONFIG_SHORT[k]] = round(a, 1)
        accs[k] = a
    row['Best']      = CONFIG_SHORT[max(accs, key=accs.get)]
    row['Worst']     = CONFIG_SHORT[min(accs, key=accs.get)]
    base = accs.get('baseline_no_rag', np.nan)
    full = accs.get('phase6_full_system', np.nan)
    row['Δ(PH6-BASE)'] = round(full - base, 1) if not (np.isnan(base) or np.isnan(full)) else np.nan
    matrix_rows.append(row)

matrix_df = pd.DataFrame(matrix_rows)

# ── Print ──────────────────────────────────────────────────────────────────
def colorize(val, base):
    if np.isnan(val): return f"{'—':>6}"
    d = val - base
    s = f"{val:5.1f}"
    if   d >= 10 : return f"\033[92m{s}\033[0m"   # bright green
    elif d >= 2  : return f"\033[96m{s}\033[0m"   # cyan
    elif d <= -10: return f"\033[91m{s}\033[0m"   # red
    elif d <= -2 : return f"\033[93m{s}\033[0m"   # yellow
    else         : return s

header = f"  {'Cty':<5} {'N':>3}  " + "  ".join(f"{s:>6}" for s in shorts) + \
         f"  {'Best':<5} {'Δ PH6':>7}"
print(header)
print("  " + "─" * (len(header) - 2))

for _, row in matrix_df.iterrows():
    base_acc = row.get('BASE', np.nan)
    cells    = "  ".join(colorize(row.get(s, np.nan), base_acc) for s in shorts)
    delta    = row['Δ(PH6-BASE)']
    d_str    = f"{delta:+.1f}" if not np.isnan(delta) else "—"
    arrow    = "▲" if not np.isnan(delta) and delta > 0 else ("▼" if not np.isnan(delta) and delta < 0 else "")
    print(f"  {row['Country']:<5} {int(row['N']):>3}  {cells}  {row['Best']:<5} {arrow}{d_str:>6}")

# ── Overall row ────────────────────────────────────────────────────────────
print("  " + "─" * (len(header) - 2))
overall = {'Country': 'ALL', 'N': len(master_df)}
cells = "  ".join(
    f"{master_df[f'correct_{k}'].mean()*100:5.1f}" for k in available
)
base_all = master_df['correct_baseline_no_rag'].mean()*100 if 'correct_baseline_no_rag' in master_df.columns else np.nan
full_all = master_df['correct_phase6_full_system'].mean()*100 if 'correct_phase6_full_system' in master_df.columns else np.nan
delta_all = full_all - base_all
print(f"  {'ALL':<5} {len(master_df):>3}  {cells}  {'—':<5} {delta_all:>+6.1f}")

# ── Summaries ─────────────────────────────────────────────────────────────
print(f"\n  📊 Best Config Frequency (# countries where each config wins):")
for cfg, cnt in matrix_df['Best'].value_counts().items():
    print(f"     {cfg:<8} {cnt:>2}x  {'█'*cnt}")

helped  = matrix_df[matrix_df['Δ(PH6-BASE)'] >  0].sort_values('Δ(PH6-BASE)', ascending=False)
hurt    = matrix_df[matrix_df['Δ(PH6-BASE)'] <  0].sort_values('Δ(PH6-BASE)')
neutral = matrix_df[matrix_df['Δ(PH6-BASE)'] == 0]

print(f"\n  ✅ PH6 HELPED ({len(helped)}): " + " ".join(f"{r['Country']}({r['Δ(PH6-BASE)']:+.0f})" for _, r in helped.iterrows()))
print(f"  ❌ PH6 HURT   ({len(hurt)}):   " + " ".join(f"{r['Country']}({r['Δ(PH6-BASE)']:+.0f})" for _, r in hurt.iterrows()))
if len(neutral): print(f"  ➖ NO CHANGE  ({len(neutral)}): " + " ".join(r['Country'] for _, r in neutral.iterrows()))

matrix_df.to_csv('/kaggle/working/country_config_matrix.csv', index=False)
print(f"\n  📁 Saved: country_config_matrix.csv")
print("=" * 85)


🌍 COUNTRY × CONFIG ACCURACY MATRIX
  Cty     N    BASE    RAG0     PH1     PH2     PH3     PH4     PH5    PH6✦  Best    Δ PH6
  ────────────────────────────────────────────────────────────────────────────────────────
  AS    2451   77.4   59.5   59.5   59.5   58.0   71.2   71.2   71.2  BASE  ▼  -6.2
  AU    513   82.7   85.6   85.6   85.6   84.0   82.7   82.7   82.7  RAG0    +0.0
  AZ    2297   79.8   79.5   79.5   79.5   78.5   81.2   81.2   81.2  PH4   ▲  +1.4
  BG    648   90.1   88.1   88.1   88.1   86.7   87.8   87.8   87.8  BASE  ▼  -2.3
  CN    1929   79.5   80.3   80.3   80.3   80.5   80.7   80.7   80.7  PH4   ▲  +1.2
  DZ    2600   82.3   80.8   80.8   80.8   79.4   79.8   79.8   79.8  BASE  ▼  -2.5
  EC    977   84.7   86.3   86.3   86.3   85.5   86.3   86.3   86.3  RAG0  ▲  +1.5
  EG    368   84.5   82.6   82.6   82.6   82.3   80.4   80.4   80.4  BASE  ▼  -4.1
  ES    1931   78.8   83.9   83.9   83.9   82.8   79.9   79.9   79.9  RAG0  ▲  +1.1
  ET    2863   59.0   63.1   63.

In [23]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL E: ALL-PAIRS COMPARISON MATRIX + McNEMAR SIGNIFICANCE
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from scipy.stats import chi2

print("=" * 85)
print("⚖️  ALL-PAIRS NET ADVANTAGE MATRIX  (row − col, * = McNemar p<0.05)")
print("=" * 85)

CONFIG_SHORT = {
    'baseline_no_rag'     : 'BASE',
    'rag_basic'           : 'RAG0',
    'phase1_countryfilter': 'PH1',
    'phase2_intent'       : 'PH2',
    'phase3_tiered'       : 'PH3',
    'phase4_quality'      : 'PH4',
    'phase5_trust_weight' : 'PH5',
    'phase6_full_system'  : 'PH6',
}

available = [k for k in CONFIG_SHORT if f'correct_{k}' in master_df.columns]
shorts    = [CONFIG_SHORT[k] for k in available]

def mcnemar_p(b, c):
    if b + c == 0: return 1.0
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    return 1 - chi2.cdf(stat, df=1)

# ── 1. Net advantage matrix ────────────────────────────────────────────────
print(f"\n  {'':>5}" + "".join(f"{s:>8}" for s in shorts))
print("  " + "─" * (5 + 8 * len(shorts)))

sig_results = []
for i, ka in enumerate(available):
    row = f"  {shorts[i]:>4} "
    for j, kb in enumerate(available):
        if i == j:
            row += f"{'—':>8}"
            continue
        a_right = master_df[f'correct_{ka}']
        b_right = master_df[f'correct_{kb}']
        b_val   = int(( a_right & ~b_right).sum())   # A right, B wrong
        c_val   = int((~a_right &  b_right).sum())   # A wrong, B right
        net     = b_val - c_val
        p       = mcnemar_p(b_val, c_val)
        star    = "*" if p < 0.05 else " "
        row    += f"{net:>+7}{star}"
        if p < 0.05 and net > 0:
            sig_results.append((shorts[i], shorts[j], net, b_val, c_val, p))
    print(row)

# ── 2. Baseline vs each phase: quadrant table ─────────────────────────────
print(f"\n  {'─'*70}")
print(f"  📊 BASELINE vs EACH CONFIG: QUADRANT BREAKDOWN")
print(f"  {'─'*70}")
print(f"  {'Config':<8} {'Both✅':>8} {'Both❌':>8} {'Fix✅':>7} {'Hurt❌':>7} {'Net':>6} {'p':>8}")
print(f"  {'─'*60}")

base_col = 'correct_baseline_no_rag'
for k in available:
    if k == 'baseline_no_rag': continue
    comp = master_df[f'correct_{k}']
    base = master_df[base_col]
    both_c = int(( base &  comp).sum())
    both_w = int((~base & ~comp).sum())
    fixed  = int((~base &  comp).sum())
    hurt   = int(( base & ~comp).sum())
    net    = fixed - hurt
    p      = mcnemar_p(fixed, hurt)
    star   = "* " if p < 0.05 else "  "
    arrow  = "▲" if net > 0 else ("▼" if net < 0 else "=")
    print(f"  {CONFIG_SHORT[k]:<8} {both_c:>8} {both_w:>8} {fixed:>7} {hurt:>7} "
          f"{arrow}{net:>+5} {p:>7.4f}{star}")

# ── 3. Significant pairs ───────────────────────────────────────────────────
if sig_results:
    print(f"\n  📊 STATISTICALLY SIGNIFICANT GAINS (p < 0.05):")
    print(f"  {'Winner':<6} > {'Loser':<6} {'Net':>6} {'Fixed':>7} {'Hurt':>6} {'p':>9}")
    print(f"  {'─'*44}")
    for w, l, net, f_, h, p in sorted(sig_results, key=lambda x: -x[2]):
        print(f"  {w:<6} > {l:<6} {net:>+6} {f_:>7} {h:>6} {p:>9.5f}")
else:
    print(f"\n  ℹ️  No pair reaches p<0.05 (n=148 is underpowered for significance).")
    print(f"     Gains are real — report effect sizes, not p-values, in the paper.")

pd.DataFrame([{
    'winner': w, 'loser': l, 'net': n_, 'fixed': f, 'hurt': h, 'p_value': p
} for w, l, n_, f, h, p in sig_results]).to_csv('/kaggle/working/sig_pairs.csv', index=False)
print(f"\n  📁 Saved: sig_pairs.csv")
print("=" * 85)


⚖️  ALL-PAIRS NET ADVANTAGE MATRIX  (row − col, * = McNemar p<0.05)

           BASE    RAG0     PH1     PH2     PH3     PH4     PH5     PH6
  ─────────────────────────────────────────────────────────────────────
  BASE        —   +293*   +293*   +293*   +467*     +4      +4      +4 
  RAG0    -293*       —     +0      +0    +174*   -289*   -289*   -289*
   PH1    -293*     +0        —     +0    +174*   -289*   -289*   -289*
   PH2    -293*     +0      +0        —   +174*   -289*   -289*   -289*
   PH3    -467*   -174*   -174*   -174*       —   -463*   -463*   -463*
   PH4      -4    +289*   +289*   +289*   +463*       —     +0      +0 
   PH5      -4    +289*   +289*   +289*   +463*     +0        —     +0 
   PH6      -4    +289*   +289*   +289*   +463*     +0      +0        —

  ──────────────────────────────────────────────────────────────────────
  📊 BASELINE vs EACH CONFIG: QUADRANT BREAKDOWN
  ──────────────────────────────────────────────────────────────────────
  Config      Bo

In [24]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL F: MULTI-DIMENSIONAL ANALYSIS
#   F1. Per-intent accuracy across configs
#   F2. Question difficulty tiers
#   F3. Ph6 remaining failure taxonomy
#   F4. Trivial vs Impossible question samples
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from collections import Counter

CONFIG_SHORT = {
    'baseline_no_rag'     : 'BASE',
    'rag_basic'           : 'RAG0',
    'phase1_countryfilter': 'PH1',
    'phase2_intent'       : 'PH2',
    'phase3_tiered'       : 'PH3',
    'phase4_quality'      : 'PH4',
    'phase5_trust_weight' : 'PH5',
    'phase6_full_system'  : 'PH6✦',
}
available = [k for k in CONFIG_SHORT if f'correct_{k}' in master_df.columns]

# ══════════════════════════════════════════════════════
# F1: PER-INTENT ACCURACY
# ══════════════════════════════════════════════════════
print("=" * 75)
print("🎯 F1: PER-INTENT ACCURACY")
print("=" * 75)

show = ['baseline_no_rag', 'phase1_countryfilter', 'phase6_full_system']
show = [k for k in show if k in available]
show_labels = [CONFIG_SHORT[k] for k in show]

intent_rows = []
for intent in sorted(master_df['intent'].unique()):
    sub = master_df[master_df['intent'] == intent]
    row = {'Intent': intent, 'N': len(sub)}
    for k in available:
        row[CONFIG_SHORT[k]] = round(sub[f'correct_{k}'].mean() * 100, 1)
    intent_rows.append(row)

intent_df = pd.DataFrame(intent_rows).sort_values('N', ascending=False)

print(f"\n  {'Intent':<25} {'N':>4}  " +
      "  ".join(f"{l:>6}" for l in show_labels) + "  Δ(PH6-BASE)")
print("  " + "─" * 60)

for _, row in intent_df.iterrows():
    vals  = "  ".join(f"{row.get(l, 0):>6.1f}" for l in show_labels)
    delta = row.get('PH6✦', 0) - row.get('BASE', 0)
    icon  = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
    print(f"  {row['Intent']:<25} {int(row['N']):>4}  {vals}  {icon}{delta:>+.1f}")

# ══════════════════════════════════════════════════════
# F2: DIFFICULTY TIERS
# ══════════════════════════════════════════════════════
print(f"\n{'='*75}")
print("📊 F2: QUESTION DIFFICULTY TIERS (by # configs correct)")
print("=" * 75)

correct_cols = [f'correct_{k}' for k in available]
master_df['n_correct'] = master_df[correct_cols].sum(axis=1)
nc = len(available)

tiers = [
    (f'Trivial    ({nc}/{nc} right)',  master_df['n_correct'] == nc),
    (f'Easy       ({nc-1}/{nc} right)',master_df['n_correct'] == nc - 1),
    (f'Medium     (4–{nc-2}/{nc})',    master_df['n_correct'].between(4, nc - 2)),
    (f'Hard       (1–3/{nc})',         master_df['n_correct'].between(1, 3)),
    (f'Impossible (0/{nc} right)',     master_df['n_correct'] == 0),
]

print(f"\n  {'Tier':<30} {'N':>5} {'%':>6}  Top countries")
print("  " + "─" * 65)
for name, mask in tiers:
    sub = master_df[mask]
    pct = len(sub) / len(master_df) * 100
    top = ", ".join(f"{c}({n})" for c,n in Counter(sub['country']).most_common(3))
    print(f"  {name:<30} {len(sub):>5} {pct:>5.1f}%  {top}")

# ══════════════════════════════════════════════════════
# F3: PH6 FAILURE TAXONOMY
# ══════════════════════════════════════════════════════
print(f"\n{'='*75}")
print("🔬 F3: PH6 FULL SYSTEM — REMAINING FAILURES")
print("=" * 75)

if 'correct_phase6_full_system' in master_df.columns:
    ph6_fail = master_df[~master_df['correct_phase6_full_system']]
    n_fail   = len(ph6_fail)
    print(f"\n  Ph6 fails on {n_fail}/{len(master_df)} ({n_fail/len(master_df)*100:.1f}%)")

    # Inherited vs introduced
    if 'correct_baseline_no_rag' in master_df.columns:
        inherited  = int((~master_df['correct_baseline_no_rag'] & ~master_df['correct_phase6_full_system']).sum())
        introduced = int(( master_df['correct_baseline_no_rag'] & ~master_df['correct_phase6_full_system']).sum())
        print(f"  ├─ Inherited (baseline also wrong):  {inherited:>3}  ({inherited/n_fail*100:.0f}%)")
        print(f"  └─ Introduced (RAG broke it):        {introduced:>3}  ({introduced/n_fail*100:.0f}%)")

    # By country
    print(f"\n  By country (error rate %):")
    ctry_fail  = ph6_fail['country'].value_counts()
    ctry_total = master_df['country'].value_counts()
    for ctry, fails in ctry_fail.head(8).items():
        rate = fails / ctry_total[ctry] * 100
        bar  = '█' * int(rate / 5)
        print(f"    {ctry:<4}  {fails}/{ctry_total[ctry]}  {rate:4.0f}%  {bar}")

    # By intent
    print(f"\n  By intent (error rate %):")
    int_fail  = ph6_fail['intent'].value_counts()
    int_total = master_df['intent'].value_counts()
    for intent, fails in int_fail.head(6).items():
        rate = fails / int_total[intent] * 100
        bar  = '█' * int(rate / 5)
        print(f"    {intent:<25}  {fails}/{int_total[intent]}  {rate:4.0f}%  {bar}")

    # Prediction bias on failures
    print(f"\n  Ph6 prediction vs Gold on failure cases:")
    gold_dist = ph6_fail['answer'].value_counts()
    pred_dist = ph6_fail['pred_phase6_full_system'].value_counts() if 'pred_phase6_full_system' in ph6_fail.columns else pd.Series()
    print(f"  {'Letter':>6}  {'Gold':>6}  {'Predicted':>10}  {'Bias':>6}")
    print(f"  {'─'*34}")
    for letter in ['A','B','C','D']:
        g = gold_dist.get(letter, 0)
        p = pred_dist.get(letter, 0)
        print(f"  {letter:>6}  {g:>6}  {p:>10}  {p-g:>+6}")

# ══════════════════════════════════════════════════════
# F4: TRIVIAL vs IMPOSSIBLE SAMPLES
# ══════════════════════════════════════════════════════
print(f"\n{'='*75}")
print("💡 F4: TRIVIAL vs IMPOSSIBLE QUESTION SAMPLES")
print("=" * 75)

trivial_mask    = master_df['n_correct'] == nc
impossible_mask = master_df['n_correct'] == 0

for label, mask in [("✅ TRIVIAL (all correct)", trivial_mask),
                    ("❌ IMPOSSIBLE (none correct)", impossible_mask)]:
    sub = master_df[mask].head(3)
    print(f"\n  {label}  — {mask.sum()} questions total")
    for _, row in sub.iterrows():
        print(f"    [{row['country']}|{row['intent']}]  {row['id']}")
        q = str(row.get('question',''))[:85]
        print(f"    Q: {q}{'...' if len(str(row.get('question',''))) > 85 else ''}")
        print(f"    Gold: {row['answer']}" +
              (f"  |  Ph6: {row.get('pred_phase6_full_system','?')}" 
               if 'pred_phase6_full_system' in row.index else ""))
        print()

# ── Save ──────────────────────────────────────────────────────────────────
intent_df.to_csv('/kaggle/working/intent_analysis.csv', index=False)
master_df.to_csv('/kaggle/working/master_df_full.csv', index=False)
print("📁 Saved: intent_analysis.csv | master_df_full.csv")
print("=" * 75)
print("✅ ALL STUDIES COMPLETE")
print("=" * 75)


🎯 F1: PER-INTENT ACCURACY

  Intent                       N    BASE     PH1    PH6✦  Δ(PH6-BASE)
  ────────────────────────────────────────────────────────────
  others                    47014    78.6    77.9    78.5  ▼-0.1

📊 F2: QUESTION DIFFICULTY TIERS (by # configs correct)

  Tier                               N      %  Top countries
  ─────────────────────────────────────────────────────────────────
  Trivial    (8/8 right)         33800  71.9%  IR(2572), GR(2153), DZ(1940)
  Easy       (7/8 right)          1615   3.4%  KP(218), ET(137), AS(110)
  Medium     (4–6/8)              2049   4.4%  AS(325), ET(165), JB(147)
  Hard       (1–3/8)              2028   4.3%  AS(282), JB(152), ET(140)
  Impossible (0/8 right)          7522  16.0%  IR(876), ET(852), JB(719)

🔬 F3: PH6 FULL SYSTEM — REMAINING FAILURES

  Ph6 fails on 10086/47014 (21.5%)
  ├─ Inherited (baseline also wrong):  8105  (80%)
  └─ Introduced (RAG broke it):        1981  (20%)

  By country (error rate %):
    ET   

In [25]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL G: RAG HURT vs FIXED vs FULL SYSTEM — COMPLETE CASE ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

print("=" * 75)
print("🔁 RAG HURT / FIXED / FULL SYSTEM — COMPLETE CASE ANALYSIS")
print("=" * 75)

# ── Shorthand columns ─────────────────────────────────────────────────────
BASE = master_df['correct_baseline_no_rag']
RAG0 = master_df['correct_rag_basic']
PH1  = master_df['correct_phase1_countryfilter']
PH6  = master_df['correct_phase6_full_system']

# ══════════════════════════════════════════════════════════════════════════
# G1: 8-WAY CATEGORY TABLE (Baseline × RAG0 × PH6)
# ══════════════════════════════════════════════════════════════════════════
print("\n📊 G1: 8-WAY OUTCOME TABLE  (Baseline × RAG Basic × Full System)")
print("─" * 65)
print(f"  {'Category':<38} {'BASE':>5} {'RAG0':>5} {'PH6':>5}  {'N':>5}  {'%':>5}")
print("─" * 65)

categories = [
    ("✅ All correct",                    True,  True,  True ),
    ("✅ Base+RAG correct, PH6 wrong",    True,  True,  False),
    ("✅ Base+PH6 correct, RAG wrong",    True,  False, True ),
    ("🔴 RAG & PH6 hurt (Base only OK)", True,  False, False),
    ("🟢 RAG+PH6 fixed (Base wrong)",    False, True,  True ),
    ("🟡 Only RAG fixed it",              False, True,  False),
    ("🔵 Only PH6 fixed it",              False, False, True ),
    ("❌ All wrong",                      False, False, False),
]

cat_masks = {}
n = len(master_df)
for label, b, r, p in categories:
    mask = (BASE == b) & (RAG0 == r) & (PH6 == p)
    cnt  = mask.sum()
    pct  = cnt / n * 100
    b_sym = "✓" if b else "✗"
    r_sym = "✓" if r else "✗"
    p_sym = "✓" if p else "✗"
    print(f"  {label:<38} {b_sym:>5} {r_sym:>5} {p_sym:>5}  {cnt:>5}  {pct:>4.1f}%")
    cat_masks[label] = mask

# ══════════════════════════════════════════════════════════════════════════
# G2: PAIRWISE QUADRANT — BASELINE vs RAG BASIC
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*75}")
print("📊 G2: BASELINE vs RAG BASIC")
print("─" * 75)

def print_quadrant(label_a, col_a, label_b, col_b):
    a = master_df[col_a]
    b = master_df[col_b]
    both_c = int(( a &  b).sum())
    both_w = int((~a & ~b).sum())
    a_fix  = int((~a &  b).sum())   # A wrong, B right → B fixed it
    a_hurt = int(( a & ~b).sum())   # A right, B wrong → B hurt it
    net    = a_fix - a_hurt
    nn     = len(master_df)
    print(f"\n  {label_b} vs {label_a}:")
    print(f"  {'Both correct':.<35} {both_c:>4}  ({both_c/nn*100:.1f}%)")
    print(f"  {'Both wrong':.<35} {both_w:>4}  ({both_w/nn*100:.1f}%)")
    print(f"  {label_b+' FIXED ('+label_a+' was wrong)':.<35} {a_fix:>4}  ({a_fix/nn*100:.1f}%)")
    print(f"  {label_b+' HURT  ('+label_a+' was right)':.<35} {a_hurt:>4}  ({a_hurt/nn*100:.1f}%)")
    arrow = "▲ BENEFIT" if net > 0 else "▼ HARM"
    print(f"  {'Net impact':.<35} {net:>+4}  {arrow}")
    return a_fix, a_hurt

rag_fixed_ids  = master_df[(~BASE) & RAG0]['id'].tolist()
rag_hurt_ids   = master_df[( BASE) & ~RAG0]['id'].tolist()
print_quadrant("Baseline", "correct_baseline_no_rag", "RAG Basic", "correct_rag_basic")

# ══════════════════════════════════════════════════════════════════════════
# G3: PAIRWISE QUADRANT — BASELINE vs FULL SYSTEM
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*75}")
print("📊 G3: BASELINE vs FULL SYSTEM (PH6)")
print("─" * 75)

ph6_fixed_ids = master_df[(~BASE) & PH6]['id'].tolist()
ph6_hurt_ids  = master_df[( BASE) & ~PH6]['id'].tolist()
print_quadrant("Baseline", "correct_baseline_no_rag", "PH6 Full", "correct_phase6_full_system")

# ══════════════════════════════════════════════════════════════════════════
# G4: RAG BASIC vs FULL SYSTEM
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*75}")
print("📊 G4: RAG BASIC vs FULL SYSTEM (did phases 1-6 improve over raw RAG?)")
print("─" * 75)
print_quadrant("RAG Basic", "correct_rag_basic", "PH6 Full", "correct_phase6_full_system")

# ══════════════════════════════════════════════════════════════════════════
# G5: DETAILED CASE BREAKDOWN — RAG HURT vs PH6 RECOVERY
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*75}")
print("📊 G5: RAG-HURT CASE RECOVERY — Did PH6 fix what RAG0 broke?")
print("─" * 75)

rag_hurt_mask = BASE & ~RAG0
recovered     = rag_hurt_mask &  PH6   # PH6 recovered it
still_hurt    = rag_hurt_mask & ~PH6   # PH6 still wrong

print(f"\n  RAG0 hurt {rag_hurt_mask.sum()} cases (baseline right, RAG wrong):")
print(f"  ✅ PH6 recovered  : {recovered.sum():>3}  ({recovered.sum()/rag_hurt_mask.sum()*100:.0f}%)")
print(f"  ❌ Still wrong    : {still_hurt.sum():>3}  ({still_hurt.sum()/rag_hurt_mask.sum()*100:.0f}%)")

# Country breakdown of still-hurt
still_hurt_df = master_df[still_hurt][['id','question','answer','country','intent',
                                        'pred_baseline_no_rag','pred_rag_basic',
                                        'pred_phase6_full_system']]
print(f"\n  Still-hurt by country:")
for ctry, cnt in still_hurt_df['country'].value_counts().items():
    print(f"    {ctry:<4}  {cnt}x")

print(f"\n  Still-hurt by intent:")
for intent, cnt in still_hurt_df['intent'].value_counts().items():
    print(f"    {intent:<25}  {cnt}x")

# ══════════════════════════════════════════════════════════════════════════
# G6: PRINT TOP-10 CASES FOR EACH CRITICAL CATEGORY
# ══════════════════════════════════════════════════════════════════════════
def print_cases(title, mask, n=5):
    sub = master_df[mask].head(n)
    if len(sub) == 0:
        return
    print(f"\n  {'─'*65}")
    print(f"  {title}  ({mask.sum()} total, showing {min(n, mask.sum())})")
    print(f"  {'─'*65}")
    for _, row in sub.iterrows():
        q = str(row.get('question', ''))[:70]
        print(f"  [{row['country']}|{row['intent']}] {row['id']}")
        print(f"  Q: {q}{'...' if len(str(row.get('question','')))>70 else ''}")
        print(f"  Gold:{row['answer']}  "
              f"BASE:{row.get('pred_baseline_no_rag','?')}  "
              f"RAG0:{row.get('pred_rag_basic','?')}  "
              f"PH6:{row.get('pred_phase6_full_system','?')}")
        print()

print(f"\n{'─'*75}")
print("📋 G6: CASE SAMPLES FOR CRITICAL CATEGORIES")
print("─" * 75)

print_cases("🔴 RAG HURT & PH6 ALSO HURT (Baseline was only correct one)",
            BASE & ~RAG0 & ~PH6, n=5)

print_cases("🟢 ONLY PH6 FIXED IT (Baseline wrong, RAG wrong, PH6 right)",
            ~BASE & ~RAG0 & PH6, n=5)

print_cases("🟡 RAG FIXED, BUT PH6 BROKE IT AGAIN (regression)",
            ~BASE & RAG0 & ~PH6, n=5)

# ══════════════════════════════════════════════════════════════════════════
# G7: SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'='*75}")
print("📋 G7: SUMMARY — NET GAINS AT EACH STAGE")
print("=" * 75)

configs_ordered = [
    ('baseline_no_rag',      'Baseline'),
    ('rag_basic',            'RAG Basic'),
    ('phase1_countryfilter', 'Ph1 Country'),
    ('phase2_intent',        'Ph2 Intent'),
    ('phase3_tiered',        'Ph3 Tiered'),
    ('phase4_quality',       'Ph4 Quality'),
    ('phase5_trust_weight',  'Ph5 Trust'),
    ('phase6_full_system',   'Ph6 Full ✦'),
]

print(f"\n  {'Config':<18} {'Acc%':>6} {'Fixed vs BASE':>14} {'Hurt vs BASE':>13} {'Net':>6}")
print("  " + "─" * 62)

base_correct = int(master_df['correct_baseline_no_rag'].sum())
for key, label in configs_ordered:
    col = f'correct_{key}'
    if col not in master_df.columns: continue
    correct = int(master_df[col].sum())
    acc = correct / n * 100
    if key == 'baseline_no_rag':
        print(f"  {label:<18} {acc:>6.1f} {'—':>14} {'—':>13} {'—':>6}")
        continue
    comp    = master_df[col]
    base    = master_df['correct_baseline_no_rag']
    fixed   = int((~base &  comp).sum())
    hurt    = int(( base & ~comp).sum())
    net     = fixed - hurt
    arrow   = "▲" if net > 0 else "▼"
    print(f"  {label:<18} {acc:>6.1f} {fixed:>14} {hurt:>13} {arrow}{net:>+5}")

# ── Save ──────────────────────────────────────────────────────────────────
still_hurt_df.to_csv('/kaggle/working/rag_still_hurt.csv', index=False)
master_df[BASE & ~RAG0 & ~PH6].to_csv('/kaggle/working/all_three_hurt.csv', index=False)
master_df[~BASE & ~RAG0 & PH6].to_csv('/kaggle/working/only_ph6_fixed.csv', index=False)
print(f"\n  📁 Saved: rag_still_hurt.csv | all_three_hurt.csv | only_ph6_fixed.csv")
print("=" * 75)


🔁 RAG HURT / FIXED / FULL SYSTEM — COMPLETE CASE ANALYSIS

📊 G1: 8-WAY OUTCOME TABLE  (Baseline × RAG Basic × Full System)
─────────────────────────────────────────────────────────────────
  Category                                BASE  RAG0   PH6      N      %
─────────────────────────────────────────────────────────────────
  ✅ All correct                              ✓     ✓     ✓  34114  72.6%
  ✅ Base+RAG correct, PH6 wrong              ✓     ✓     ✗    715   1.5%
  ✅ Base+PH6 correct, RAG wrong              ✓     ✗     ✓    837   1.8%
  🔴 RAG & PH6 hurt (Base only OK)            ✓     ✗     ✗   1266   2.7%
  🟢 RAG+PH6 fixed (Base wrong)               ✗     ✓     ✓   1335   2.8%
  🟡 Only RAG fixed it                        ✗     ✓     ✗    475   1.0%
  🔵 Only PH6 fixed it                        ✗     ✗     ✓    642   1.4%
  ❌ All wrong                                ✗     ✗     ✗   7630  16.2%

───────────────────────────────────────────────────────────────────────────
📊 G2: BASEL

In [26]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL H: ORACLE RAG UPPER BOUND (#6) + KB COVERAGE PER COUNTRY (#15)
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
from collections import defaultdict

print("=" * 75)
print("🔭 ORACLE RAG UPPER BOUND + KB COVERAGE ANALYSIS")
print("=" * 75)

# ══════════════════════════════════════════════════════════════════════════
# H1: KB CHUNKS PER COUNTRY
# ══════════════════════════════════════════════════════════════════════════
print("\n📊 H1: KB CHUNKS PER COUNTRY")
print("─" * 55)

kb_by_country = defaultdict(list)
for chunk in kb_chunks:
    ctry = str(chunk.get('country', chunk.get('source_country', ''))).upper().strip()
    if not ctry or ctry == 'NONE':
        # Try to infer from source URL
        src = str(chunk.get('source', chunk.get('url', '')))
        ctry = 'UNK'
    kb_by_country[ctry].append(chunk)

kb_count_df = pd.DataFrame([
    {'country': k, 'kb_chunks': len(v)} for k, v in kb_by_country.items()
]).sort_values('kb_chunks', ascending=False)

# Merge with per-country accuracy (baseline + PH6)
ctry_acc = master_df.groupby('country').agg(
    n_questions            = ('id', 'count'),
    acc_baseline           = ('correct_baseline_no_rag', 'mean'),
    acc_ph6                = ('correct_phase6_full_system', 'mean'),
).reset_index()
ctry_acc['acc_baseline'] *= 100
ctry_acc['acc_ph6']      *= 100

kb_corr_df = ctry_acc.merge(kb_count_df, on='country', how='left').fillna({'kb_chunks': 0})
kb_corr_df['kb_chunks'] = kb_corr_df['kb_chunks'].astype(int)
kb_corr_df['rag_gain']  = kb_corr_df['acc_ph6'] - kb_corr_df['acc_baseline']

print(f"\n  {'Country':<6} {'Questions':>10} {'KB Chunks':>10} {'BASE%':>7} {'PH6%':>7} {'RAG Gain':>9}")
print("  " + "─" * 55)
for _, row in kb_corr_df.sort_values('kb_chunks', ascending=False).iterrows():
    gain_arrow = "▲" if row['rag_gain'] > 0 else ("▼" if row['rag_gain'] < 0 else "=")
    print(f"  {row['country']:<6} {int(row['n_questions']):>10} {int(row['kb_chunks']):>10} "
          f"{row['acc_baseline']:>6.1f}% {row['acc_ph6']:>6.1f}% "
          f"{gain_arrow}{row['rag_gain']:>+7.1f}%")

# Pearson correlation: kb_chunks vs rag_gain
valid = kb_corr_df[kb_corr_df['kb_chunks'] > 0]
if len(valid) > 3:
    from scipy.stats import pearsonr
    r, p = pearsonr(valid['kb_chunks'], valid['rag_gain'])
    print(f"\n  📈 Pearson r (KB chunks vs RAG gain): r={r:.3f}, p={p:.4f}")
    print(f"  {'✅ More KB → better RAG' if r > 0 else '❌ More KB does NOT help RAG'}")

# ══════════════════════════════════════════════════════════════════════════
# H2: ORACLE RAG UPPER BOUND
# For each question: does the correct answer appear in any KB chunk?
# ══════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*75}")
print("📊 H2: ORACLE RAG UPPER BOUND")
print("─" * 75)
print("  Checking if correct answer text exists anywhere in KB...\n")

# Build fast lookup: country → list of chunk texts
kb_text_by_country = defaultdict(list)
for chunk in kb_chunks:
    text = str(chunk.get('page_content', chunk.get('text', ''))).lower()
    ctry = str(chunk.get('country', '')).upper().strip()
    kb_text_by_country[ctry].append(text)
    kb_text_by_country['ALL'].append(text)  # global fallback

oracle_rows = []
for _, row in master_df.iterrows():
    answer_letter  = row['answer']
    opt_col        = f'option_{answer_letter}'
    answer_text    = str(row.get(opt_col, '')).lower().strip()
    ctry           = row['country']

    if not answer_text or answer_text in ['', 'nan']:
        oracle_rows.append({'id': row['id'], 'country': ctry, 
                            'intent': row['intent'], 'in_kb': False, 'search': 'no_text'})
        continue

    # Check country-specific chunks first, then all
    found = False
    chunks_to_search = kb_text_by_country.get(ctry, []) + kb_text_by_country.get('ALL', [])
    for chunk_text in chunks_to_search[:500]:  # cap for speed
        if answer_text[:30] in chunk_text:      # first 30 chars of answer
            found = True
            break

    oracle_rows.append({'id': row['id'], 'country': ctry,
                        'intent': row['intent'], 'in_kb': found})

oracle_df = pd.DataFrame(oracle_rows)
n_in_kb   = oracle_df['in_kb'].sum()
n_total   = len(oracle_df)
coverage  = n_in_kb / n_total * 100

print(f"  Questions answerable from KB : {n_in_kb}/{n_total} ({coverage:.1f}%)")
print(f"  Questions NOT in KB (ceiling): {n_total - n_in_kb}/{n_total} ({100-coverage:.1f}%)")
print(f"\n  Theoretical RAG ceiling      : {coverage:.1f}%  (only if retrieval + LLM perfect)")
print(f"  Actual PH6 accuracy          : {master_df['correct_phase6_full_system'].mean()*100:.1f}%")
print(f"  Gap to ceiling               : {coverage - master_df['correct_phase6_full_system'].mean()*100:.1f}pp")

# Coverage by country
print(f"\n  Oracle coverage by country:")
for ctry, grp in oracle_df.groupby('country'):
    cov = grp['in_kb'].mean() * 100
    bar = '█' * int(cov / 10)
    print(f"    {ctry:<4}  {int(grp['in_kb'].sum())}/{len(grp)}  ({cov:4.0f}%)  {bar}")

# Coverage by intent
print(f"\n  Oracle coverage by intent:")
for intent, grp in oracle_df.groupby('intent'):
    cov = grp['in_kb'].mean() * 100
    print(f"    {intent:<25}  {int(grp['in_kb'].sum())}/{len(grp)}  ({cov:4.0f}%)")

# ── Save ──────────────────────────────────────────────────────────────────
oracle_df.to_csv('/kaggle/working/oracle_coverage.csv', index=False)
kb_corr_df.to_csv('/kaggle/working/kb_country_coverage.csv', index=False)
print(f"\n  📁 Saved: oracle_coverage.csv | kb_country_coverage.csv")
print("=" * 75)


🔭 ORACLE RAG UPPER BOUND + KB COVERAGE ANALYSIS

📊 H1: KB CHUNKS PER COUNTRY
───────────────────────────────────────────────────────

  Country  Questions  KB Chunks   BASE%    PH6%  RAG Gain
  ───────────────────────────────────────────────────────
  AS           2451         43   77.4%   71.2% ▼   -6.2%
  CN           1929         42   79.5%   80.7% ▲   +1.2%
  ES           1931         42   78.8%   79.9% ▲   +1.1%
  IR           3699         41   73.2%   73.0% ▼   -0.2%
  US           1942         37   93.2%   90.4% ▼   -2.7%
  GB           2167         36   92.4%   89.7% ▼   -2.7%
  ID           1995         34   79.8%   81.2% ▲   +1.4%
  ET           2863         34   59.0%   63.0% ▲   +4.1%
  AZ           2297         34   79.8%   81.2% ▲   +1.4%
  GR           2734         34   83.2%   83.1% ▼   -0.2%
  KR           2512         34   84.0%   83.4% ▼   -0.6%
  NG           2008         34   68.9%   69.5% ▲   +0.6%
  DZ           2600         33   82.3%   79.8% ▼   -2.5%
  KP     

In [27]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL I: COUNTRY-INTENT HEATMAP (#12) + WEB PRESENCE CORRELATION (#5)
# ═══════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np

print("=" * 75)
print("🗺️  COUNTRY × INTENT HEATMAP + WEB PRESENCE CORRELATION")
print("=" * 75)

# ══════════════════════════════════════════════════════════════════════════
# I1: COUNTRY × INTENT ACCURACY HEATMAP (text-based)
# ══════════════════════════════════════════════════════════════════════════
print("\n📊 I1: COUNTRY × INTENT ACCURACY HEATMAP (PH6 accuracy %)")
print("─" * 75)
print("  Color: 🟩 ≥80%  🟨 50–79%  🟥 <50%  · = no data\n")

countries = sorted(master_df['country'].unique())
intents   = sorted(master_df['intent'].unique())

# Build pivot
heatmap = master_df.groupby(['country','intent'])['correct_phase6_full_system'].agg(
    ['mean','count']
).reset_index()
heatmap['acc'] = heatmap['mean'] * 100
pivot = heatmap.pivot(index='country', columns='intent', values='acc')
count_pivot = heatmap.pivot(index='country', columns='intent', values='count').fillna(0)

# Print header
intent_short = {i: i[:6] for i in intents}
header = f"  {'Ctry':<6}" + "".join(f"{intent_short[i]:>8}" for i in intents)
print(header)
print("  " + "─" * len(header))

for ctry in countries:
    row_str = f"  {ctry:<6}"
    for intent in intents:
        val = pivot.loc[ctry, intent] if ctry in pivot.index and intent in pivot.columns else np.nan
        cnt = int(count_pivot.loc[ctry, intent]) if ctry in count_pivot.index else 0
        if np.isnan(val) or cnt == 0:
            row_str += f"{'·':>8}"
        elif val >= 80:
            row_str += f"\033[92m{val:>7.0f}%\033[0m"
        elif val >= 50:
            row_str += f"\033[93m{val:>7.0f}%\033[0m"
        else:
            row_str += f"\033[91m{val:>7.0f}%\033[0m"
    print(row_str)

# Hardest country-intent combos
print(f"\n  🔴 5 HARDEST country-intent combos (min 2 questions):")
hard = heatmap[heatmap['count'] >= 2].nsmallest(5, 'acc')
for _, r in hard.iterrows():
    print(f"    {r['country']} × {r['intent']:<25}  {r['acc']:5.1f}%  (n={int(r['count'])})")

print


🗺️  COUNTRY × INTENT HEATMAP + WEB PRESENCE CORRELATION

📊 I1: COUNTRY × INTENT ACCURACY HEATMAP (PH6 accuracy %)
───────────────────────────────────────────────────────────────────────────
  Color: 🟩 ≥80%  🟨 50–79%  🟥 <50%  · = no data

  Ctry    others
  ────────────────
  AS         71%
  AU         83%
  AZ         81%
  BG         88%
  CN         81%
  DZ         80%
  EC         86%
  EG         80%
  ES         80%
  ET         63%
  FR         87%
  GB         90%
  GR         83%
  ID         81%
  IE         88%
  IR         73%
  JB         63%
  JP         81%
  KP         66%
  KR         83%
  LK         90%
  MA         77%
  MX         94%
  NG         70%
  PH         78%
  PV         79%
  SA         74%
  SE         80%
  SG         88%
  US         90%

  🔴 5 HARDEST country-intent combos (min 2 questions):
    ET × others                      63.0%  (n=2863)
    JB × others                      63.2%  (n=2345)
    KP × others                      66.2%  (n=2185)
 

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [28]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL J: PHASE 9 - STATISTICAL SIGNIFICANCE (McNemar's Test & Cohen's h)
# ═══════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from scipy.stats import chi2
import math

print("=" * 75)
print("📈 PHASE 9: STATISTICAL SIGNIFICANCE TESTING")
print("=" * 75)

def mcnemar_test(preds_a, preds_b, gold):
    """Computes McNemar's test for paired predictions."""
    b, c = 0, 0
    for pa, pb, g in zip(preds_a, preds_b, gold):
        if pa == g and pb != g: b += 1 # A correct, B wrong
        if pa != g and pb == g: c += 1 # A wrong, B correct
    
    if b + c == 0:
        return 0.0, 1.0, False
        
    # Chi-square statistic with continuity correction
    statistic = (abs(b - c) - 1)**2 / (b + c)
    p_value = 1 - chi2.cdf(statistic, df=1)
    return statistic, p_value, p_value < 0.05

def cohens_h(acc_a, acc_b):
    """Calculates Cohen's h effect size for two proportions."""
    h = 2 * math.asin(math.sqrt(acc_b)) - 2 * math.asin(math.sqrt(acc_a))
    abs_h = abs(h)
    if abs_h < 0.2: interpretation = "Small"
    elif abs_h < 0.5: interpretation = "Medium"
    else: interpretation = "Large"
    return h, interpretation

# Prepare data
gold_list = [gold_map[qid] for qid in master_df['id']]
configs_to_test = [
    ('baseline_no_rag', 'Baseline'),
    ('rag_basic', 'Basic RAG'),
    ('phase6_full_system', 'Full System')
]

results = []
for i in range(len(configs_to_test)):
    for j in range(i + 1, len(configs_to_test)):
        key_a, name_a = configs_to_test[i]
        key_b, name_b = configs_to_test[j]
        
        preds_a = predictions[key_a]['predicted'].tolist()
        preds_b = predictions[key_b]['predicted'].tolist()
        
        acc_a = master_df[f'correct_{key_a}'].mean()
        acc_b = master_df[f'correct_{key_b}'].mean()
        
        stat, p_val, is_sig = mcnemar_test(preds_a, preds_b, gold_list)
        h_val, h_interp = cohens_h(acc_a, acc_b)
        
        results.append({
            'System_A': name_a, 'System_B': name_b,
            'Delta_%': (acc_b - acc_a) * 100,
            'p_value': p_val, 'Significant': is_sig,
            'Cohens_h': abs(h_val), 'Effect_Size': h_interp
        })

sig_df = pd.DataFrame(results)

print(f"{'System A':<15} {'System B':<15} {'Delta':>8} {'p-value':>10} {'Sig?':>6} {'Cohen h':>10} {'Effect'}")
print("-" * 75)
for _, r in sig_df.iterrows():
    sig_mark = "✓" if r['Significant'] else "✗"
    print(f"{r['System_A']:<15} {r['System_B']:<15} {r['Delta_%']:>7.1f}% {r['p_value']:>10.4f} {sig_mark:>5} {r['Cohens_h']:>10.3f}  {r['Effect_Size']}")

sig_df.to_csv('/kaggle/working/statistical_significance.csv', index=False)
print("\n📁 Saved: statistical_significance.csv")


📈 PHASE 9: STATISTICAL SIGNIFICANCE TESTING
System A        System B           Delta    p-value   Sig?    Cohen h Effect
---------------------------------------------------------------------------
Baseline        Basic RAG          -0.6%     0.0000     ✓      0.015  Small
Baseline        Full System        -0.0%     0.9620     ✗      0.000  Small
Basic RAG       Full System         0.6%     0.0000     ✓      0.015  Small

📁 Saved: statistical_significance.csv


In [29]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL K: PHASE 10 - LATEX TABLES & SUBMISSION PACKAGING
# ═══════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

print("=" * 75)
print("📝 PHASE 10: LATEX TABLES & ARTIFACT GENERATION")
print("=" * 75)

# 1. GENERATE LATEX: Ablation Table
ablation_latex = r"""\begin{table}[h]
\centering
\begin{tabular}{lccc}
\toprule
\textbf{Configuration} & \textbf{Accuracy (\%)} & \textbf{$\Delta$ vs Prev} & \textbf{Net vs Base} \\
\midrule
"""
base_acc = master_df['correct_baseline_no_rag'].mean() * 100
prev_acc = base_acc

ablation_configs = [
    ('baseline_no_rag', 'Baseline (No RAG)'),
    ('rag_basic', 'Basic RAG'),
    ('phase1_countryfilter', '+ Country Filter'),
    ('phase2_intent', '+ Intent Tagging'),
    ('phase3_tiered', '+ Tiered Routing'),
    ('phase4_quality', '+ Anti-Leak Prompts'),
    ('phase5_trust_weight', '+ Trust Weights'),
    ('phase6_full_system', 'Full System (Phase 6)')
]

for key, name in ablation_configs:
    if f'correct_{key}' not in master_df: continue
    acc = master_df[f'correct_{key}'].mean() * 100
    delta_prev = acc - prev_acc
    net_base = acc - base_acc
    
    if key == 'baseline_no_rag':
        ablation_latex += f"{name} & {acc:.1f} & -- & -- \\\\\n"
    else:
        ablation_latex += f"{name} & {acc:.1f} & {delta_prev:+.1f} & {net_base:+.1f} \\\\\n"
    prev_acc = acc

ablation_latex += r"""\bottomrule
\end{tabular}
\caption{Ablation Study Component Contributions.}
\label{tab:ablation}
\end{table}"""

# 2. GENERATE LATEX: Heatmap (Top 5 hardest)
heatmap_latex = r"""\begin{table}[h]
\centering
\begin{tabular}{llc}
\toprule
\textbf{Country} & \textbf{Intent} & \textbf{Accuracy (\%)} \\
\midrule
"""
heatmap = master_df.groupby(['country','intent'])['correct_phase6_full_system'].agg(['mean','count']).reset_index()
hardest = heatmap[heatmap['count'] >= 2].nsmallest(5, 'mean')
for _, r in hardest.iterrows():
    heatmap_latex += f"{r['country']} & {r['intent'].replace('_', ' ')} & {r['mean']*100:.1f} \\\\\n"

heatmap_latex += r"""\bottomrule
\end{tabular}
\caption{Top 5 Hardest Country-Intent Combinations.}
\label{tab:hardest_intents}
\end{table}"""

# Write to file
with open("/kaggle/working/paper_tables.tex", "w") as f:
    f.write("% --- ABLATION TABLE ---\n")
    f.write(ablation_latex)
    f.write("\n\n% --- HARDEST INTENTS TABLE ---\n")
    f.write(heatmap_latex)

print("✅ LaTeX tables generated: paper_tables.tex")
print("\nPreview of Ablation Table LaTeX:")
print(ablation_latex)

# 3. GENERATE SEMEVAL SUBMISSION TSV
print("\n" + "─" * 75)
print("📦 GENERATING FINAL SEMEVAL SUBMISSION TSV")

sub_df = predictions['phase6_full_system'].copy()
sub_file = "/kaggle/working/submission_final.tsv"
sub_df.to_csv(sub_file, sep='\t', index=False, header=False)

print(f"✅ Submission formatted and saved: {sub_file}")
print(f"   Shape: {sub_df.shape} (Must be exactly 148 rows for Dev set)")
print(f"   Format: No header, Tab-separated, [id, prediction]")
print("=" * 75)


📝 PHASE 10: LATEX TABLES & ARTIFACT GENERATION
✅ LaTeX tables generated: paper_tables.tex

Preview of Ablation Table LaTeX:
\begin{table}[h]
\centering
\begin{tabular}{lccc}
\toprule
\textbf{Configuration} & \textbf{Accuracy (\%)} & \textbf{$\Delta$ vs Prev} & \textbf{Net vs Base} \\
\midrule
Baseline (No RAG) & 78.6 & -- & -- \\
Basic RAG & 77.9 & -0.6 & -0.6 \\
+ Country Filter & 77.9 & +0.0 & -0.6 \\
+ Intent Tagging & 77.9 & +0.0 & -0.6 \\
+ Tiered Routing & 77.6 & -0.4 & -1.0 \\
+ Anti-Leak Prompts & 78.5 & +1.0 & -0.0 \\
+ Trust Weights & 78.5 & +0.0 & -0.0 \\
Full System (Phase 6) & 78.5 & +0.0 & -0.0 \\
\bottomrule
\end{tabular}
\caption{Ablation Study Component Contributions.}
\label{tab:ablation}
\end{table}

───────────────────────────────────────────────────────────────────────────
📦 GENERATING FINAL SEMEVAL SUBMISSION TSV
✅ Submission formatted and saved: /kaggle/working/submission_final.tsv
   Shape: (47014, 2) (Must be exactly 148 rows for Dev set)
   Format: No header, 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 1: LINE CHART – Ablation Accuracy Progression
# CHART 2: BAR CHART – System Accuracy Comparison
# ═══════════════════════════════════════════════════════════════════════════

# --- Compute per-config accuracy ---
ablation_accs = []
for cfg in CONFIGS_ORDERED:
    col = f'correct_{cfg}'
    acc = master_df[col].mean() * 100
    ablation_accs.append({
        'config': cfg,
        'label': CONFIG_LABELS[cfg],
        'accuracy': acc,
    })

abl_df = pd.DataFrame(ablation_accs)
abl_df['delta'] = abl_df['accuracy'] - abl_df['accuracy'].iloc[0]

# --- CHART 1: LINE CHART – Ablation Progression ---
fig_line = go.Figure()
fig_line.add_trace(go.Scatter(
    x=abl_df['label'],
    y=abl_df['accuracy'],
    mode='lines+markers+text',
    text=[f"{a:.2f}%" for a in abl_df['accuracy']],
    textposition='top center',
    marker=dict(size=10, color=[CONFIG_COLORS[c] for c in CONFIGS_ORDERED]),
    line=dict(width=3, color='#636EFA'),
    hovertemplate='%{x}<br>Accuracy: %{y:.2f}%<extra></extra>'
))
fig_line.update_layout(
    title=dict(text='Ablation Accuracy Progression Across 8 Configurations',
               font=dict(size=18)),
    xaxis_title='Configuration',
    yaxis_title='Accuracy (%)',
    yaxis=dict(range=[min(abl_df['accuracy'])-1, max(abl_df['accuracy'])+1]),
    xaxis_tickangle=-30,
    height=500, width=900,
    margin=dict(b=120),
)
fig_line.show()

# --- CHART 2: BAR CHART – System Accuracy Comparison ---
fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=abl_df['label'],
    y=abl_df['accuracy'],
    marker_color=[CONFIG_COLORS[c] for c in CONFIGS_ORDERED],
    text=[f"{a:.2f}%" for a in abl_df['accuracy']],
    textposition='outside',
    hovertemplate='%{x}<br>Accuracy: %{y:.2f}%<extra></extra>'
))
fig_bar.update_layout(
    title=dict(text='System Accuracy Comparison (8 Ablation Configs)',
               font=dict(size=18)),
    xaxis_title='Configuration',
    yaxis_title='Accuracy (%)',
    yaxis=dict(range=[0, max(abl_df['accuracy'])+3]),
    xaxis_tickangle=-30,
    height=500, width=900,
    margin=dict(b=120),
)
fig_bar.show()

print("📊 Chart 1: Ablation line progression")
print("📊 Chart 2: System accuracy bar comparison")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 3: GROUPED BAR – Baseline vs RAG vs Full System by Country (Top 15)
# CHART 4: STACKED BAR – Error Composition by Intent Category
# ═══════════════════════════════════════════════════════════════════════════

# --- CHART 3: GROUPED BAR – Three Systems by Country ---
ctry_compare = master_df.groupby('country').agg(
    n=('id', 'count'),
    baseline=('correct_baseline_no_rag', 'mean'),
    rag_basic=('correct_rag_basic', 'mean'),
    full_system=('correct_phase6_full_system', 'mean'),
).reset_index()
ctry_compare[['baseline','rag_basic','full_system']] *= 100

# Top 15 by question count
top15 = ctry_compare.nlargest(15, 'n').sort_values('full_system', ascending=False)

fig_grouped = go.Figure()
fig_grouped.add_trace(go.Bar(
    name='Baseline (No RAG)', x=top15['country'], y=top15['baseline'],
    marker_color='#636EFA', text=[f"{v:.1f}" for v in top15['baseline']],
    textposition='outside', textfont_size=9
))
fig_grouped.add_trace(go.Bar(
    name='Basic RAG', x=top15['country'], y=top15['rag_basic'],
    marker_color='#EF553B', text=[f"{v:.1f}" for v in top15['rag_basic']],
    textposition='outside', textfont_size=9
))
fig_grouped.add_trace(go.Bar(
    name='Full System (PH6)', x=top15['country'], y=top15['full_system'],
    marker_color='#B6E880', text=[f"{v:.1f}" for v in top15['full_system']],
    textposition='outside', textfont_size=9
))
fig_grouped.update_layout(
    title=dict(text='Baseline vs RAG vs Full System – Top 15 Countries by Volume',
               font=dict(size=16)),
    xaxis_title='Country', yaxis_title='Accuracy (%)',
    barmode='group',
    yaxis=dict(range=[0, max(top15[['baseline','rag_basic','full_system']].max()) + 8]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    height=550, width=1000,
)
fig_grouped.show()

# --- CHART 4: STACKED BAR – Error Composition by Intent ---
intent_acc = master_df.groupby('intent').agg(
    n=('id', 'count'),
    correct_base=('correct_baseline_no_rag', 'sum'),
    correct_ph6=('correct_phase6_full_system', 'sum'),
).reset_index()
intent_acc['wrong_base'] = intent_acc['n'] - intent_acc['correct_base']
intent_acc['wrong_ph6'] = intent_acc['n'] - intent_acc['correct_ph6']
intent_acc['fixed_by_ph6'] = (
    (master_df.groupby('intent')
     .apply(lambda g: ((g['correct_baseline_no_rag'] == 0) & (g['correct_phase6_full_system'] == 1)).sum())
     .reset_index(drop=True))
).values if len(intent_acc) == len(master_df['intent'].unique()) else 0

# Recompute properly
fixed_data = []
for intent in master_df['intent'].unique():
    mask = master_df['intent'] == intent
    n = mask.sum()
    both_correct = ((master_df.loc[mask, 'correct_baseline_no_rag'] == 1) & 
                    (master_df.loc[mask, 'correct_phase6_full_system'] == 1)).sum()
    only_base = ((master_df.loc[mask, 'correct_baseline_no_rag'] == 1) & 
                 (master_df.loc[mask, 'correct_phase6_full_system'] == 0)).sum()
    only_ph6 = ((master_df.loc[mask, 'correct_baseline_no_rag'] == 0) & 
                (master_df.loc[mask, 'correct_phase6_full_system'] == 1)).sum()
    both_wrong = ((master_df.loc[mask, 'correct_baseline_no_rag'] == 0) & 
                  (master_df.loc[mask, 'correct_phase6_full_system'] == 0)).sum()
    fixed_data.append({
        'intent': intent, 'n': n,
        'Both Correct': both_correct, 'Only Baseline': only_base,
        'Fixed by PH6': only_ph6, 'Both Wrong': both_wrong
    })
stacked_df = pd.DataFrame(fixed_data).sort_values('n', ascending=True)

fig_stacked = go.Figure()
for cat, color in [('Both Correct', '#00CC96'), ('Fixed by PH6', '#B6E880'),
                   ('Only Baseline', '#FFA15A'), ('Both Wrong', '#EF553B')]:
    fig_stacked.add_trace(go.Bar(
        name=cat, y=stacked_df['intent'], x=stacked_df[cat],
        orientation='h', marker_color=color,
        hovertemplate=f'{cat}<br>Intent: %{{y}}<br>Count: %{{x}}<extra></extra>'
    ))
fig_stacked.update_layout(
    title=dict(text='Error Composition by Intent: Baseline vs Full System',
               font=dict(size=16)),
    xaxis_title='Number of Questions', yaxis_title='Intent Category',
    barmode='stack',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    height=500, width=1000,
    margin=dict(l=200),
)
fig_stacked.show()

print("📊 Chart 3: Grouped bar – 3 systems × top 15 countries")
print("📊 Chart 4: Stacked bar – error composition by intent")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 5: HORIZONTAL BAR – Per-Country RAG Gain/Loss (PH6 vs Baseline)
# CHART 6: SCATTER – KB Coverage vs RAG Accuracy Gain
# CHART 7: BUBBLE – Country Accuracy (size = question count)
# ═══════════════════════════════════════════════════════════════════════════

# --- Build per-country stats ---
ctry_stats = master_df.groupby('country').agg(
    n=('id', 'count'),
    acc_base=('correct_baseline_no_rag', 'mean'),
    acc_ph6=('correct_phase6_full_system', 'mean'),
).reset_index()
ctry_stats['acc_base'] *= 100
ctry_stats['acc_ph6'] *= 100
ctry_stats['rag_delta'] = ctry_stats['acc_ph6'] - ctry_stats['acc_base']

# --- CHART 5: HORIZONTAL BAR – RAG Gain/Loss per Country ---
hbar_df = ctry_stats.sort_values('rag_delta', ascending=True)
colors = ['#EF553B' if d < 0 else '#00CC96' for d in hbar_df['rag_delta']]

fig_hbar = go.Figure()
fig_hbar.add_trace(go.Bar(
    y=hbar_df['country'], x=hbar_df['rag_delta'],
    orientation='h', marker_color=colors,
    text=[f"{d:+.2f}%" for d in hbar_df['rag_delta']],
    textposition='outside',
    hovertemplate='%{y}<br>RAG Δ: %{x:+.2f}%<br>Base: %{customdata[0]:.1f}%<br>PH6: %{customdata[1]:.1f}%<extra></extra>',
    customdata=hbar_df[['acc_base', 'acc_ph6']].values
))
fig_hbar.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)
fig_hbar.update_layout(
    title=dict(text='Per-Country RAG Gain/Loss (Full System vs Baseline)',
               font=dict(size=16)),
    xaxis_title='Accuracy Delta (pp)', yaxis_title='Country',
    height=700, width=900,
    margin=dict(l=60),
)
fig_hbar.show()

# --- CHART 6: SCATTER – KB Coverage vs RAG Gain ---
# Use kb_corr_df (computed in Cell H) if available, else recompute
try:
    scatter_df = kb_corr_df.copy()
except NameError:
    # Recompute KB chunks per country
    from collections import defaultdict
    kb_by_country = defaultdict(int)
    for chunk in kb_chunks:
        ctry = str(chunk.get('country', chunk.get('source_country', ''))).upper().strip()
        if ctry and ctry != 'NONE':
            kb_by_country[ctry] += 1
    kb_count_df = pd.DataFrame([
        {'country': k, 'kb_chunks': v} for k, v in kb_by_country.items()
    ])
    scatter_df = ctry_stats.merge(kb_count_df, on='country', how='left').fillna({'kb_chunks': 0})
    scatter_df['rag_gain'] = scatter_df['rag_delta']

fig_scatter = px.scatter(
    scatter_df, x='kb_chunks', y='rag_delta' if 'rag_delta' in scatter_df.columns else 'rag_gain',
    text='country',
    color='rag_delta' if 'rag_delta' in scatter_df.columns else 'rag_gain',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    labels={'kb_chunks': 'KB Chunks Available', 
            'rag_delta': 'RAG Accuracy Gain (pp)',
            'rag_gain': 'RAG Accuracy Gain (pp)'},
    title='KB Coverage vs RAG Accuracy Gain per Country',
)
fig_scatter.update_traces(
    textposition='top center', marker=dict(size=12),
    textfont_size=9
)
fig_scatter.update_layout(height=550, width=900)
fig_scatter.show()

# --- CHART 7: BUBBLE – Country Accuracy (size = question count) ---
fig_bubble = px.scatter(
    ctry_stats, x='acc_base', y='acc_ph6',
    size='n', text='country',
    color='rag_delta',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    size_max=40,
    labels={'acc_base': 'Baseline Accuracy (%)',
            'acc_ph6': 'Full System Accuracy (%)',
            'n': 'Question Count',
            'rag_delta': 'RAG Δ (pp)'},
    title='Country Accuracy: Baseline vs Full System (bubble size = # questions)',
)
fig_bubble.update_traces(textposition='top center', textfont_size=8)
# Add diagonal reference line (y=x)
xy_min = min(ctry_stats['acc_base'].min(), ctry_stats['acc_ph6'].min()) - 2
xy_max = max(ctry_stats['acc_base'].max(), ctry_stats['acc_ph6'].max()) + 2
fig_bubble.add_trace(go.Scatter(
    x=[xy_min, xy_max], y=[xy_min, xy_max],
    mode='lines', line=dict(dash='dash', color='gray', width=1),
    showlegend=False, hoverinfo='skip'
))
fig_bubble.update_layout(height=600, width=800)
fig_bubble.show()

print("📊 Chart 5: Horizontal bar – per-country RAG gain/loss")
print("📊 Chart 6: Scatter – KB coverage vs accuracy gain")
print("📊 Chart 7: Bubble – country accuracy (size = question count)")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 8: HISTOGRAM – Difficulty Distribution (n_correct across 8 configs)
# CHART 9: BOX PLOT – Per-Country Accuracy Spread by Config
# CHART 10: VIOLIN – Accuracy Distribution by Config
# ═══════════════════════════════════════════════════════════════════════════

# --- CHART 8: HISTOGRAM – Difficulty Distribution ---
fig_hist = px.histogram(
    master_df, x='n_correct',
    nbins=9,   # 0 through 8
    color_discrete_sequence=['#636EFA'],
    labels={'n_correct': '# Configs Answering Correctly (out of 8)', 'count': 'Questions'},
    title='Question Difficulty Distribution (how many of 8 configs got it right)',
)
fig_hist.update_layout(
    bargap=0.05,
    xaxis=dict(dtick=1),
    yaxis_title='Number of Questions',
    height=450, width=800,
)
# Add difficulty tier annotations
fig_hist.add_annotation(x=0, y=0, yref='paper', yshift=10,
    text='Impossible', showarrow=False, font=dict(size=10, color='red'))
fig_hist.add_annotation(x=8, y=0, yref='paper', yshift=10,
    text='Trivial', showarrow=False, font=dict(size=10, color='green'))
fig_hist.show()

# --- CHART 9: BOX PLOT – Per-Country Accuracy by Config ---
# Build long-form DataFrame: one row per (country, config) with accuracy
box_rows = []
for cfg in CONFIGS_ORDERED:
    col = f'correct_{cfg}'
    ctry_acc = master_df.groupby('country')[col].mean() * 100
    for ctry, acc in ctry_acc.items():
        box_rows.append({'config': CONFIG_LABELS[cfg], 'country': ctry, 'accuracy': acc})
box_df = pd.DataFrame(box_rows)

fig_box = px.box(
    box_df, x='config', y='accuracy',
    color='config',
    color_discrete_sequence=list(CONFIG_COLORS.values()),
    labels={'accuracy': 'Accuracy (%)', 'config': 'Configuration'},
    title='Per-Country Accuracy Spread for Each Configuration',
    points='all',
)
fig_box.update_layout(
    showlegend=False,
    xaxis_tickangle=-30,
    height=550, width=1000,
    margin=dict(b=140),
)
fig_box.show()

# --- CHART 10: VIOLIN – Accuracy Distribution by Config ---
fig_violin = px.violin(
    box_df, x='config', y='accuracy',
    color='config',
    color_discrete_sequence=list(CONFIG_COLORS.values()),
    box=True, points='all',
    labels={'accuracy': 'Accuracy (%)', 'config': 'Configuration'},
    title='Accuracy Distribution Across Countries (Violin Plot)',
)
fig_violin.update_layout(
    showlegend=False,
    xaxis_tickangle=-30,
    height=550, width=1000,
    margin=dict(b=140),
)
fig_violin.show()

print("📊 Chart 8: Histogram – difficulty distribution")
print("📊 Chart 9: Box plot – accuracy spread by config")
print("📊 Chart 10: Violin – accuracy distribution by config")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 11: HEATMAP – Country × Config Accuracy Matrix
# CHART 12: CONFUSION MATRICES – Baseline & Full System (Predicted vs Gold)
# ═══════════════════════════════════════════════════════════════════════════

# --- CHART 11: HEATMAP – Country × Config Accuracy Matrix ---
heatmap_data = {}
for cfg in CONFIGS_ORDERED:
    col = f'correct_{cfg}'
    heatmap_data[CONFIG_LABELS[cfg]] = master_df.groupby('country')[col].mean() * 100

heat_df = pd.DataFrame(heatmap_data)
heat_df = heat_df.sort_index()  # sort countries alphabetically

fig_heatmap = go.Figure(data=go.Heatmap(
    z=heat_df.values,
    x=[label.replace('Phase ', 'PH') for label in heat_df.columns],
    y=heat_df.index,
    colorscale='RdYlGn',
    zmid=heat_df.values.mean(),
    text=np.round(heat_df.values, 1),  
    texttemplate='%{text:.1f}',
    textfont=dict(size=8),
    hovertemplate='Country: %{y}<br>Config: %{x}<br>Accuracy: %{z:.2f}%<extra></extra>',
    colorbar=dict(title='Acc %'),
))
fig_heatmap.update_layout(
    title=dict(text='Country × Configuration Accuracy Heatmap (%)',
               font=dict(size=16)),
    xaxis_title='Configuration',
    yaxis_title='Country',
    height=800, width=1000,
    yaxis=dict(autorange='reversed'),
    margin=dict(l=60),
)
fig_heatmap.show()

# --- CHART 12: CONFUSION MATRICES – Predicted vs Gold (A/B/C/D) ---
answer_labels = ['A', 'B', 'C', 'D']

def build_confusion_matrix(config_key):
    """Build 4×4 confusion matrix: rows=Gold, cols=Predicted."""
    pred_col = f'pred_{config_key}'
    cm = np.zeros((4, 4), dtype=int)
    for _, row in master_df.iterrows():
        gold_idx = answer_labels.index(row['answer']) if row['answer'] in answer_labels else -1
        pred_idx = answer_labels.index(row[pred_col]) if row[pred_col] in answer_labels else -1
        if gold_idx >= 0 and pred_idx >= 0:
            cm[gold_idx][pred_idx] += 1
    return cm

# Baseline confusion matrix
cm_base = build_confusion_matrix('baseline_no_rag')
# Full system confusion matrix
cm_ph6 = build_confusion_matrix('phase6_full_system')

fig_cm = make_subplots(rows=1, cols=2,
                       subplot_titles=['Baseline (No RAG)', 'Full System (Phase 6)'],
                       horizontal_spacing=0.12)

for i, (cm, col) in enumerate([(cm_base, 1), (cm_ph6, 2)]):
    # Normalize by row for percentages
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    
    fig_cm.add_trace(go.Heatmap(
        z=cm_pct,
        x=answer_labels, y=answer_labels,
        colorscale='Blues',
        text=np.array([[f"{cm[r][c]}\n({cm_pct[r][c]:.0f}%)" 
                        for c in range(4)] for r in range(4)]),
        texttemplate='%{text}',
        textfont=dict(size=10),
        showscale=(col == 2),
        hovertemplate='Gold: %{y}<br>Predicted: %{x}<br>Count: %{z:.0f}%<extra></extra>',
        colorbar=dict(title='Row %') if col == 2 else None,
    ), row=1, col=col)

fig_cm.update_xaxes(title_text='Predicted', row=1, col=1)
fig_cm.update_xaxes(title_text='Predicted', row=1, col=2)
fig_cm.update_yaxes(title_text='Gold Answer', row=1, col=1)
fig_cm.update_yaxes(title_text='Gold Answer', row=1, col=2)
fig_cm.update_layout(
    title=dict(text='Confusion Matrix: Predicted vs Gold Answer (A/B/C/D)',
               font=dict(size=16)),
    height=450, width=900,
)
fig_cm.show()

print("📊 Chart 11: Heatmap – country × config accuracy")
print("📊 Chart 12: Confusion matrices – baseline vs full system")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 13: PIE – Gold Answer Distribution (A/B/C/D)
# CHART 14: DONUT – Intent Category Distribution
# CHART 15: AREA – Cumulative Accuracy Gain Across Ablation Phases
# ═══════════════════════════════════════════════════════════════════════════

# --- CHART 13: PIE – Gold Answer Distribution ---
gold_dist = master_df['answer'].value_counts().sort_index()

fig_pie = go.Figure(data=[go.Pie(
    labels=gold_dist.index,
    values=gold_dist.values,
    hole=0,
    textinfo='label+percent+value',
    texttemplate='%{label}<br>%{value:,} (%{percent:.1%})',
    marker=dict(colors=['#636EFA', '#EF553B', '#00CC96', '#AB63FA']),
    hovertemplate='Answer %{label}<br>Count: %{value:,}<br>Pct: %{percent:.1%}<extra></extra>',
)])
fig_pie.update_layout(
    title=dict(text='Gold Answer Distribution (A/B/C/D) – 47,014 Questions',
               font=dict(size=16)),
    height=450, width=550,
)
fig_pie.show()

# --- CHART 14: DONUT – Intent Category Distribution ---
intent_dist = master_df['intent'].value_counts().sort_values(ascending=False)

fig_donut = go.Figure(data=[go.Pie(
    labels=intent_dist.index,
    values=intent_dist.values,
    hole=0.45,
    textinfo='label+percent',
    texttemplate='%{label}<br>%{percent:.1%}',
    hovertemplate='%{label}<br>Count: %{value:,}<br>%{percent:.1%}<extra></extra>',
)])
fig_donut.add_annotation(
    text=f'{len(intent_dist)}<br>Intents',
    x=0.5, y=0.5, font_size=16, showarrow=False
)
fig_donut.update_layout(
    title=dict(text='Question Intent Category Distribution (Donut)',
               font=dict(size=16)),
    height=500, width=650,
)
fig_donut.show()

# --- CHART 15: AREA – Cumulative Accuracy Gain Across Phases ---
abl_accs_area = []
for cfg in CONFIGS_ORDERED:
    col = f'correct_{cfg}'
    acc = master_df[col].mean() * 100
    abl_accs_area.append({'config': CONFIG_LABELS[cfg], 'accuracy': acc})

area_df = pd.DataFrame(abl_accs_area)
area_df['delta_vs_base'] = area_df['accuracy'] - area_df['accuracy'].iloc[0]
area_df['cumulative_improvement'] = area_df['delta_vs_base'].clip(lower=0).cummax()

fig_area = go.Figure()
fig_area.add_trace(go.Scatter(
    x=area_df['config'], y=area_df['accuracy'],
    fill='tozeroy',
    mode='lines+markers',
    line=dict(color='#636EFA', width=2),
    fillcolor='rgba(99, 110, 250, 0.2)',
    name='Accuracy',
    text=[f"{a:.2f}%" for a in area_df['accuracy']],
    hovertemplate='%{x}<br>Accuracy: %{y:.2f}%<extra></extra>',
))
fig_area.add_trace(go.Scatter(
    x=area_df['config'], y=area_df['delta_vs_base'],
    fill='tozeroy',
    mode='lines+markers',
    line=dict(color='#00CC96', width=2),
    fillcolor='rgba(0, 204, 150, 0.2)',
    name='Δ vs Baseline (pp)',
    hovertemplate='%{x}<br>Δ vs Baseline: %{y:+.2f}pp<extra></extra>',
))
fig_area.update_layout(
    title=dict(text='Accuracy & Cumulative Gain Across Ablation Phases',
               font=dict(size=16)),
    xaxis_title='Configuration', yaxis_title='Value',
    xaxis_tickangle=-30,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    height=500, width=950,
    margin=dict(b=130),
)
fig_area.show()

print("📊 Chart 13: Pie – gold answer distribution")
print("📊 Chart 14: Donut – intent category distribution")
print("📊 Chart 15: Area – cumulative accuracy gain across phases")
print("\n✅ ALL 15 PLOTLY CHARTS ADDED SUCCESSFULLY")
